In [1]:
# ==========================
# 1. Imports
# ==========================
from pathlib import Path
import platform

import duckdb
import pandas as pd

# ==========================
# 1. Import
# ==========================
import numpy as np


# ==========================
# 2. Parâmetros
# ==========================
GESTAO = "LIS"

ANOS = [
    2025,
    2026,
]


# ==========================
# 3. Pasta das bases
# ==========================
def get_db_dir():

    sistema = platform.system()

    if sistema == "Windows":
        return Path(
            r"C:\Users\LISARR\OneDrive - Salvesen Logística S.A\python\00.DB"
        )

    if sistema == "Darwin":
        return Path(
            "/Users/rr/Library/Mobile Documents/"
            "com~apple~CloudDocs/05.Salvesen/00_DB"
        )

    return Path.cwd()


DB_DIR = get_db_dir()


# ==========================
# 4. Caminhos
# ==========================
PATHS = {
    ano: DB_DIR / f"{ano}.duckdb"
    for ano in ANOS
}


# ==========================
# 5. Validar bases
# ==========================
for ano, path in PATHS.items():

    if not path.exists():
        raise FileNotFoundError(
            f"Base {ano} não encontrada: {path}"
        )


# ==========================
# 6. Ligações DuckDB
# ==========================
CON = {
    ano: duckdb.connect(
        str(PATHS[ano]),
        read_only=True
    )
    for ano in ANOS
}


# ==========================
# 7. Validar
# ==========================
validacao = pd.DataFrame({
    "Ano": ANOS,
    "Base": [
        PATHS[ano].name
        for ano in ANOS
    ],
    "Tabelas": [
        len(CON[ano].sql("SHOW TABLES").fetchall())
        for ano in ANOS
    ],
})

validacao

,Ano,Base,Tabelas
0,2025,2025.duckdb,9
1,2026,2026.duckdb,11


In [2]:
# ==========================
# 1. Configuração
# ==========================
meses = {
    1: "Jan",
    2: "Fev",
    3: "Mar",
    4: "Abr",
    5: "Mai",
    6: "Jun",
    7: "Jul",
    8: "Ago",
    9: "Set",
    10: "Out",
    11: "Nov",
    12: "Dez",
}

tabelas_paletes = {}


# ==========================
# 2. Processar anos
# ==========================
for ano in ANOS:

    df_ano = CON[ano].execute(f"""
        SELECT
            CODACT,
            ACTIVIDAD AS CLIENTE,
            FENTREGA,
            TRY_CAST(
                REPLACE(PALETS, ',', '.')
                AS DOUBLE
            ) AS PALETS
        FROM inform_27_{ano}
        WHERE Gestion = ?
    """, [GESTAO]).df()

    # ==========================
    # 3. Extrair mês
    # ==========================
    df_ano["MES"] = pd.to_numeric(
        df_ano["FENTREGA"]
        .astype("string")
        .str.strip()
        .str[4:6],
        errors="coerce"
    )

    # ==========================
    # 4. Criar tabela mensal
    # ==========================
    tabela = (
        df_ano
        .pivot_table(
            index=["CODACT", "CLIENTE"],
            columns="MES",
            values="PALETS",
            aggfunc="sum",
            fill_value=0
        )
        .reindex(
            columns=range(1, 13),
            fill_value=0
        )
        .rename(columns=meses)
        .reset_index()
    )

    # ==========================
    # 5. Formatar paletes
    # ==========================
    colunas_meses = list(meses.values())

    tabela[colunas_meses] = (
        tabela[colunas_meses]
        .round(0)
        .astype(int)
    )

    tabelas_paletes[ano] = tabela


# ==========================
# 6. Mostrar 2026
# ==========================
tabelas_paletes[2026]

MES,CODACT,CLIENTE,Jan,Fev,Mar,Abr,Mai,Jun,Jul,Ago,Set,Out,Nov,Dez
0,011,DANONE PORTUGAL,2939,2473,14378,15000,16625,15909,16358,18342,3509,0,0,0
1,013,DANONE PORTUGAL CAPILAR,2982,2914,4166,3990,3609,3173,3365,3046,656,0,0,0
2,015,TROPICANA,22,8,109,92,154,256,383,234,55,0,0,0
3,028,PLN,0,0,0,0,1,9,18,25,6,0,0,0
4,068,FRIESLANDCAMPIÑA,1,0,7,0,15,0,0,0,0,0,0,0
5,074,NUTRICIA,1298,858,5599,4934,4221,3778,4169,4181,1164,0,0,0
6,084,DANONE PORTUGAL TLD,495,363,528,495,528,1695,807,528,132,0,0,0
7,116,ORANGINA SCHWEPPES ESPAÑA BASE,0,0,0,0,3,6,5,8,0,0,0,0
8,117,ORANGINA SCHWEPPES,0,8,72,88,84,192,288,170,23,0,0,0
9,118,ORANGNA SCHWEPPES PORTUGAL,59,45,482,447,422,501,594,588,121,0,0,0


In [3]:
# ==========================
# 1. Configuração
# ==========================
ano = 2025

meses = {
    1: "Jan",
    2: "Fev",
    3: "Mar",
    4: "Abr",
    5: "Mai",
    6: "Jun",
    7: "Jul",
    8: "Ago",
    9: "Set",
    10: "Out",
    11: "Nov",
    12: "Dez",
}


# ==========================
# 2. Ler dados
# ==========================
df_2025 = CON[ano].execute(f"""
    SELECT
        CODACT,
        ACTIVIDAD AS CLIENTE,
        FENTREGA,
        TRY_CAST(
            REPLACE(PALETS, ',', '.')
            AS DOUBLE
        ) AS PALETS
    FROM inform_27_{ano}
    WHERE Gestion = ?
""", [GESTAO]).df()


# ==========================
# 3. Extrair mês
# ==========================
df_2025["MES"] = pd.to_numeric(
    df_2025["FENTREGA"]
    .astype("string")
    .str.strip()
    .str[4:6],
    errors="coerce"
)


# ==========================
# 4. Criar tabela mensal
# ==========================
tabela_paletes_2025 = (
    df_2025
    .pivot_table(
        index=["CODACT", "CLIENTE"],
        columns="MES",
        values="PALETS",
        aggfunc="sum",
        fill_value=0
    )
    .reindex(
        columns=range(1, 13),
        fill_value=0
    )
    .rename(columns=meses)
    .reset_index()
)


# ==========================
# 5. Formatar paletes
# ==========================
colunas_meses = list(meses.values())

tabela_paletes_2025[colunas_meses] = (
    tabela_paletes_2025[colunas_meses]
    .round(0)
    .astype(int)
)


# ==========================
# 6. Mostrar
# ==========================
tabela_paletes_2025

MES,CODACT,CLIENTE,Jan,Fev,Mar,Abr,Mai,Jun,Jul,Ago,Set,Out,Nov,Dez
0,011,DANONE PORTUGAL,7358,7457,6611,7720,7963,7816,9843,5863,2787,8690,8777,2943
1,013,DANONE PORTUGAL CAPILAR,1692,1550,1229,900,1574,1658,1793,877,1439,3211,3883,2649
2,015,TROPICANA,63,78,64,84,146,182,277,141,37,120,106,30
3,068,FRIESLANDCAMPIÑA,2,0,1,0,0,0,2,4,3,10,5,2
4,074,NUTRICIA,3294,2464,2770,2867,3583,3298,3990,1807,996,3247,3260,1464
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,711,XPO HELM,93,108,58,45,54,88,41,19,58,80,0,0
63,757,SUMOLCOMPAL MARKETING,82,89,39,83,46,90,236,86,108,228,119,17
64,784,SABORES DO VEZ,40,45,40,33,77,37,49,16,53,29,30,34
65,860,ANDROS PTG CONGELADO,0,0,1,2,0,11,0,0,0,0,0,0


In [4]:
# ==========================
# 1. Validar cliente 011
# ==========================
validacao_011 = CON[2025].execute("""
    SELECT
        SUBSTRING(FENTREGA, 5, 2) AS MES,
        COUNT(*) AS LINHAS,
        COUNT(DISTINCT CODEUT) AS VIAGENS,
        COUNT(DISTINCT FENTREGA) AS DIAS,
        MIN(FENTREGA) AS PRIMEIRA_ENTREGA,
        MAX(FENTREGA) AS ULTIMA_ENTREGA,
        ROUND(
            SUM(
                TRY_CAST(
                    REPLACE(PALETS, ',', '.')
                    AS DOUBLE
                )
            ),
            0
        ) AS PALETS
    FROM inform_27_2025
    WHERE Gestion = 'LIS'
      AND CODACT = '011'
    GROUP BY
        SUBSTRING(FENTREGA, 5, 2)
    ORDER BY
        MES
""").df()


# ==========================
# 2. Mostrar
# ==========================
validacao_011

,MES,LINHAS,VIAGENS,DIAS,PRIMEIRA_ENTREGA,ULTIMA_ENTREGA,PALETS
0,01,1500,575,25,20250102,20250131,7358.0
1,02,1423,501,24,20250201,20250228,7457.0
2,03,1283,496,26,20250301,20250331,6611.0
3,04,1559,465,26,20250401,20250430,7720.0
4,05,1514,498,27,20250501,20250531,7963.0
5,06,1435,489,25,20250602,20250630,7816.0
6,07,2032,682,26,20250701,20250731,9843.0
7,08,1153,469,24,20250801,20250829,5863.0
8,09,713,273,23,20250901,20250930,2787.0
9,10,1557,624,27,20251001,20251031,8690.0


In [7]:
# ==========================
# 1. Imports
# ==========================
import numpy as np
import pandas as pd


# ==========================
# 2. Parâmetros
# ==========================
GESTAO = "LIS"

MESES = {
    1: "Jan",
    2: "Fev",
    3: "Mar",
    4: "Abr",
    5: "Mai",
    6: "Jun",
    7: "Jul",
    8: "Ago",
    9: "Set",
    10: "Out",
    11: "Nov",
    12: "Dez",
}


# ==========================
# 3. Função de leitura
# ==========================
def ler_paletes(ano):

    return CON[ano].execute(f"""
        SELECT
            TRIM(CAST(CODACT AS VARCHAR)) AS CODACT,
            MODE(ACTIVIDAD) AS CLIENTE,
            CAST(SUBSTRING(FENTREGA, 5, 2) AS INTEGER) AS MES,
            SUM(
                TRY_CAST(
                    REPLACE(PALETS, ',', '.')
                    AS DOUBLE
                )
            ) AS PALETS
        FROM inform_27_{ano}
        WHERE Gestion = ?
        GROUP BY
            CODACT,
            MES
    """, [GESTAO]).df()


# ==========================
# 4. Normalizar CODACT
# ==========================
def normalizar_codact(valor):

    valor = str(valor).strip().upper()

    if valor.isdigit():
        return str(int(valor))

    return valor


# ==========================
# 5. Ler dados
# ==========================
dados_2025 = ler_paletes(2025)
dados_2026 = ler_paletes(2026)

dados_2025["CODACT"] = dados_2025["CODACT"].apply(normalizar_codact)
dados_2026["CODACT"] = dados_2026["CODACT"].apply(normalizar_codact)


# ==========================
# 6. Criar tabelas mensais
# ==========================
def criar_tabela(df):

    return (
        df
        .pivot_table(
            index=["CODACT", "CLIENTE"],
            columns="MES",
            values="PALETS",
            aggfunc="sum",
            fill_value=0
        )
        .reindex(
            columns=range(1, 13),
            fill_value=0
        )
        .rename(columns=MESES)
        .reset_index()
    )


t25 = criar_tabela(dados_2025)
t26 = criar_tabela(dados_2026)


# ==========================
# 7. Usar clientes de 2026
# ==========================
base = t26.merge(
    t25.drop(columns="CLIENTE"),
    on="CODACT",
    how="left",
    suffixes=("_26", "_25")
)

base = base.fillna(0)


# ==========================
# 8. Calcular evolução
# ==========================
meses_base = [
    "Jan",
    "Fev",
    "Mar",
    "Abr",
    "Mai",
    "Jun",
    "Jul",
    "Ago",
]

total_25 = base[
    [f"{m}_25" for m in meses_base]
].sum(axis=1)

total_26 = base[
    [f"{m}_26" for m in meses_base]
].sum(axis=1)

base["factor"] = np.where(
    total_25 > 0,
    total_26 / total_25,
    np.nan
)


# ==========================
# 9. Prever meses futuros
# ==========================
media_2026 = base[
    [f"{m}_26" for m in meses_base]
].mean(axis=1)

base["Out_prev"] = np.where(
    base["factor"].notna(),
    base["Out_25"] * base["factor"],
    media_2026
)

base["Nov_prev"] = np.where(
    base["factor"].notna(),
    base["Nov_25"] * base["factor"],
    media_2026
)

dez_25_tratado = (
    base["Out_25"]
    + base["Nov_25"]
) / 2

base["Dez_prev"] = np.where(
    base["factor"].notna(),
    dez_25_tratado * base["factor"],
    media_2026
)


# ==========================
# 10. Tabela final
# ==========================
resultado = pd.DataFrame({
    "CODACT": base["CODACT"],
    "CLIENTE": base["CLIENTE"],

    "Jan": base["Jan_26"],
    "Fev": base["Fev_26"],
    "Mar": base["Mar_26"],
    "Abr": base["Abr_26"],
    "Mai": base["Mai_26"],
    "Jun": base["Jun_26"],
    "Jul": base["Jul_26"],
    "Ago": base["Ago_26"],
    "Set": base["Set_26"],

    "Out Prev.": base["Out_prev"],
    "Nov Prev.": base["Nov_prev"],
    "Dez Prev.": base["Dez_prev"],
})


# ==========================
# 11. Formatar
# ==========================
colunas_valores = resultado.columns[2:]

resultado[colunas_valores] = (
    resultado[colunas_valores]
    .round(0)
    .astype(int)
)

resultado = (
    resultado
    .sort_values(
        "Jan",
        ascending=False
    )
    .reset_index(drop=True)
)


# ==========================
# 12. Mostrar
# ==========================
resultado

,CODACT,CLIENTE,Jan,Fev,Mar,Abr,Mai,Jun,Jul,Ago,Set,Out Prev.,Nov Prev.,Dez Prev.
0,13,DANONE PORTUGAL CAPILAR,2982,2914,4166,3990,3609,3173,3365,3046,656,7760,9384,8572
1,11,DANONE PORTUGAL,2939,2473,14378,15000,16625,15909,16358,18342,3509,14623,14769,14696
2,74,NUTRICIA,1298,858,5599,4934,4221,3778,4169,4181,1164,3917,3932,3924
3,311,ALPRO PORTUGAL,1191,982,3118,3676,3295,3180,3805,3970,902,3310,3351,3331
4,260,ANDROS REFRIGERADO,689,538,2564,2506,2701,2047,2289,2009,463,3501,3008,3254
5,429,H-3 Carnes,559,394,1477,1500,1636,1595,1760,1778,310,1287,1566,1426
6,84,DANONE PORTUGAL TLD,495,363,528,495,528,1695,807,528,132,942,955,949
7,131,RANA,344,253,1259,1249,1383,1161,1260,1220,276,1794,1672,1733
8,401,LUIS VICENTE,317,324,346,378,225,2,1,0,0,283,165,224
9,273,LACTOGAL,304,222,1330,1223,661,0,0,0,0,2307,2052,2179
